# POPPER-Swarm Demo: Automated Study Validation & Reproduction with Agentic Sequential Falsifications

This notebook demonstrates how to use `Popper` (adapted from [snap-stanford/POPPER](https://github.com/snap-stanford/POPPER), Huang et al., ICML 2025) together with **Exa Web Search** and **past.dev Multi-Agent Memory** to validate biological hypotheses and reproduce raw-FASTQ sequencing studies on **BixBench3**.

In [1]:
from popper import Popper, __version__
from popper.benchmark import gene_perturb_hypothesis

print(f"Popper version: {__version__}")
agent = Popper(llm="claude-3-5-sonnet-20240620")
agent.configure(
    alpha=0.05,
    max_num_of_tests=6,
    max_retry=3,
    aggregate_test="E-value",
    relevance_checker=True,
    use_react_agent=True,
    use_exa_search=True,
    use_past_memory=True,
)
print(f"Ville rejection threshold (alpha=0.05): E_t >= {1 / 0.05:.1f}")

Popper version: 0.1.0
Ville rejection threshold (alpha=0.05): E_t >= 20.0


In [2]:
import json
bm = gene_perturb_hypothesis(num_of_samples=19, permuted=False, dataset="TAGSEQ", path="outputs")
example = bm.get_example(1)
print("Loaded BixBench3 Hypothesis Example:")
print(json.dumps(example, indent=2))

Loaded BixBench3 Hypothesis Example:
{
  "task_id": 9,
  "artifact_name": "t98g_c10_vs_control_degs.csv",
  "prompt": "Artifact t98g_c10_vs_control_degs.csv satisfies all biological and structural invariants of Task 9.",
  "round": 1,
  "e_value": 78.125,
  "binary_answer": false
}


In [3]:
results = agent.validate(
    hypothesis="REST knockout in T98G clone C10 derepresses canonical neuronal markers (SNAP25, SYP, SYN1, BDNF)."
)
print("Validation Result:")
print(json.dumps({k: v for k, v in results.items() if k != 'sub_tests'}, indent=2))

Validation Result:
{
  "main_hypothesis": "REST knockout in T98G clone C10 derepresses canonical neuronal markers (SNAP25, SYP, SYN1, BDNF).",
  "falsification_test_result": "Validated: survived 3 falsification checks (E_t=0.1494 < 20.0)",
  "e_value_product": 0.149429,
  "alpha": 0.05,
  "ville_threshold": 20.0,
  "rejected": false,
  "conclusion": true
}
